# ColorCheck adaptive deterministic matcher

Fit palette and tone strengths from TST100K expert edits. The current browser matcher is the baseline; only preflight training and validation images are opened. The final partition stays sealed. This is a GPU-assisted pilot, not a deployment decision.

In [ ]:
from pathlib import Path
import numpy, sklearn, torch
for directory in ('tools', 'tools/ml', 'tools/ml/tst100k'):
    path = Path('/kaggle/working') / directory
    path.mkdir(parents=True, exist_ok=True)
    (path / '__init__.py').touch()
print({'numpy': numpy.__version__, 'sklearn': sklearn.__version__, 'torch': torch.__version__})


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/data.py
"""Select and stage a small, scene-disjoint TST100K triplet experiment.

The upstream manifest is the pairing authority. Scene IDs, rather than rows,
determine the split: a photograph used as either content or reference cannot
appear in a different split. Images are downloaded only inside Kaggle scratch.
"""

from __future__ import annotations

import hashlib
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from threading import Lock
from urllib.parse import quote
from urllib.request import urlopen
from urllib.error import HTTPError, URLError

import torch
import numpy as np
from PIL import Image, ImageOps
from torch.utils.data import Dataset

DATASET_REVISION = "63bf13833b75407c862d608c3d81281d0a35b61f"
BASE_URL = f"https://huggingface.co/datasets/ToneStyle/TST100K/resolve/{DATASET_REVISION}"
SPLITS = ("train", "validation", "test")


class RequestPacer:
    """Space image requests and share server backoff across worker threads."""

    def __init__(self, interval_seconds: float = 0.75):
        self.interval_seconds = interval_seconds
        self.next_request_at = 0.0
        self.lock = Lock()

    def wait(self) -> None:
        while True:
            with self.lock:
                remaining = self.next_request_at - time.monotonic()
                if remaining <= 0:
                    self.next_request_at = time.monotonic() + self.interval_seconds
                    return
            time.sleep(remaining)

    def pause_for(self, seconds: float) -> None:
        with self.lock:
            self.next_request_at = max(self.next_request_at, time.monotonic() + seconds)


def scene_id(path: str) -> str:
    """Map source and preset-rendered copies to one original photograph."""
    return Path(path).stem.split("_")[0]


def scene_split(identifier: str, seed: int) -> str:
    """Hash each photograph once; references cannot cross a split boundary."""
    digest = hashlib.sha256(f"{seed}:{identifier}".encode()).digest()
    bucket = int.from_bytes(digest[:8], "big") % 10
    return "train" if bucket < 8 else "validation" if bucket == 8 else "test"


def select_triplets(
    records: list[dict[str, str]], counts: tuple[int, int, int], seed: int
) -> dict[str, list[dict[str, str]]]:
    """Select a reproducible sample with distinct target scenes in each split.

    Triplets whose content and reference belong to different partitions are
    omitted. This costs data but prevents direct scene reuse across evaluation.
    """
    selected = {split: [] for split in SPLITS}
    used_content = {split: set() for split in SPLITS}
    ranked = sorted(
        records,
        key=lambda row: hashlib.sha256(
            f"{seed}:{row['content']}:{row['reference']}:{row['gt']}".encode()
        ).digest(),
    )
    for row in ranked:
        if set(row) != {"content", "reference", "gt"}:
            raise ValueError("Each TST100K record must have content, reference, and gt")
        content_id = scene_id(row["content"])
        reference_id = scene_id(row["reference"])
        if content_id == reference_id or scene_id(row["gt"]) != content_id:
            continue
        split = scene_split(content_id, seed)
        if scene_split(reference_id, seed) != split:
            continue
        if content_id in used_content[split] or len(selected[split]) >= counts[SPLITS.index(split)]:
            continue
        selected[split].append(row)
        used_content[split].add(content_id)
        if all(len(selected[name]) == counts[index] for index, name in enumerate(SPLITS)):
            break
    actual = tuple(len(selected[name]) for name in SPLITS)
    if actual != counts:
        raise ValueError(f"Insufficient disjoint triplets: requested {counts}, found {actual}")
    return selected


def fetch_manifest() -> list[dict[str, str]]:
    """Read the pinned upstream pairing manifest without storing it in Git."""
    with urlopen(f"{BASE_URL}/triplet.json", timeout=120) as response:
        return json.load(response)


def _download_one(root: Path, relative: str, max_bytes: int, pacer: RequestPacer) -> int:
    """Stream one image to Kaggle scratch, with bounded size and retries."""
    destination = root / relative
    if destination.is_file():
        return destination.stat().st_size
    destination.parent.mkdir(parents=True, exist_ok=True)
    url = f"{BASE_URL}/{quote(relative, safe='/')}"
    for attempt in range(7):
        temporary = destination.with_name(destination.name + ".part")
        try:
            pacer.wait()
            with urlopen(url, timeout=90) as response:
                size = 0
                with temporary.open("wb") as output:
                    while chunk := response.read(1024 * 1024):
                        size += len(chunk)
                        if size > max_bytes:
                            raise ValueError(f"Image exceeds {max_bytes} bytes: {relative}")
                        output.write(chunk)
            with Image.open(temporary) as image:
                image.verify()
            temporary.replace(destination)
            return size
        except HTTPError as error:
            temporary.unlink(missing_ok=True)
            if error.code not in (429, 502, 503, 504) or attempt == 6:
                raise
            retry_after = error.headers.get("Retry-After")
            try:
                delay = max(1.0, float(retry_after)) if retry_after else 0.0
            except ValueError:
                delay = 0.0
            delay = min(300.0, max(delay, 20.0 * 2**attempt))
            pacer.pause_for(delay)
            print(f"Data host HTTP {error.code}; pausing image requests for {delay:.0f}s", flush=True)
        except (URLError, OSError, ValueError) as error:
            temporary.unlink(missing_ok=True)
            if attempt == 6 or isinstance(error, ValueError):
                raise
            time.sleep(min(30, 2**attempt))
    raise RuntimeError("Unreachable download retry state")


def stage_images(
    selected: dict[str, list[dict[str, str]]], root: Path, workers: int = 2
) -> dict[str, int]:
    """Download only selected files; the full 94 GB collection is never staged."""
    paths = sorted({path for rows in selected.values() for row in rows for path in row.values()})
    total_bytes = 0
    pacer = RequestPacer()
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = {pool.submit(_download_one, root, path, 40_000_000, pacer): path for path in paths}
        for completed, task in enumerate(as_completed(pending), 1):
            total_bytes += task.result()
            if completed % 250 == 0 or completed == len(paths):
                print(json.dumps({"downloadedFiles": completed, "totalFiles": len(paths)}), flush=True)
    return {"files": len(paths), "bytes": total_bytes}


class TripletDataset(Dataset):
    """Decode aligned source/answer pixels and an independent style reference."""

    def __init__(self, rows: list[dict[str, str]], root: Path, size: int = 128):
        self.rows = rows
        self.root = root
        self.size = size

    def __len__(self) -> int:
        return len(self.rows)

    def _image(self, path: str) -> torch.Tensor:
        with Image.open(self.root / path) as loaded:
            image = ImageOps.exif_transpose(loaded).convert("RGB")
            image = image.resize((self.size, self.size), Image.Resampling.BILINEAR)
            pixels = torch.from_numpy(np.array(image, copy=True))
        return pixels.permute(2, 0, 1).float().div_(255)

    def __getitem__(self, index: int) -> dict:
        row = self.rows[index]
        return {
            "source": self._image(row["content"]),
            "reference": self._image(row["reference"]),
            "expected": self._image(row["gt"]),
            "scene": scene_id(row["content"]),
        }


def save_split_manifest(selected: dict, path: Path, seed: int) -> None:
    """Record every pairing and revision needed to reproduce the experiment."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps({"datasetRevision": DATASET_REVISION, "seed": seed, "splits": selected}, indent=2)
    )


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/prepare_v4.py
"""Freeze v4's final holdout and verify strong-edit validation coverage.

Run this on Kaggle CPU before training. It excludes every scene used by v3 or
its later audit, then checks that validation can test the stated hypothesis.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

from torch.utils.data import DataLoader

from .data import TripletDataset, fetch_manifest, scene_id, stage_images

FINAL_SEED = 6109
FINAL_COUNT = 256
STRONG_EDIT_THRESHOLD = 0.08


def mounted_file(kernel_slug: str, name: str) -> Path:
    """Resolve a mounted output without confusing two similarly named files."""
    matches = [
        path for path in Path("/kaggle/input").rglob(name)
        if kernel_slug in path.parts
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {name} from {kernel_slug}; found {matches}")
    return matches[0]


def select_final_rows(records: list[dict], old_splits: dict, audit_rows: list[dict]) -> list[dict]:
    """Select fixed, never-seen target and reference scenes by hash rank."""
    used = {
        scene_id(row[key])
        for rows in (*old_splits.values(), audit_rows)
        for row in rows
        for key in ("content", "reference")
    }
    ranked = sorted(
        records,
        key=lambda row: hashlib.sha256(
            f"{FINAL_SEED}:{row['content']}:{row['reference']}:{row['gt']}".encode()
        ).digest(),
    )
    selected = []
    targets = set()
    for row in ranked:
        target = scene_id(row["content"])
        reference = scene_id(row["reference"])
        if (
            target in used or reference in used or target == reference
            or target in targets or scene_id(row["gt"]) != target
        ):
            continue
        selected.append(row)
        targets.add(target)
        if len(selected) == FINAL_COUNT:
            break
    if len(selected) != FINAL_COUNT:
        raise ValueError(f"Expected {FINAL_COUNT} untouched targets; found {len(selected)}")
    return selected


def strong_edit_count(rows: list[dict], scratch: Path) -> int:
    """Count source/expert pairs with at least 0.08 mean RGB change."""
    loader = DataLoader(TripletDataset(rows, scratch, 128), batch_size=8, num_workers=2)
    return sum(
        int(((batch["expected"] - batch["source"]).abs().mean(dim=(1, 2, 3))
            >= STRONG_EDIT_THRESHOLD).sum())
        for batch in loader
    )


def main() -> None:
    output = Path("/kaggle/working/colorcheck-tst100k-v4-preflight")
    scratch = Path("/kaggle/temp/colorcheck-tst100k-v4-preflight")
    output.mkdir(parents=True, exist_ok=True)
    old = json.loads(mounted_file("colorcheck-tst100k-expanded-validation", "splits.json").read_text())
    audit = json.loads(mounted_file("colorcheck-tst100k-fresh-evaluation", "splits.json").read_text())
    final_rows = select_final_rows(fetch_manifest(), old["splits"], audit["rows"])
    payload = {"datasetRevision": old["datasetRevision"], "seed": FINAL_SEED, "rows": final_rows}
    manifest = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    (output / "final-holdout.json").write_text(json.dumps(payload, indent=2))
    checksum = hashlib.sha256(manifest.encode()).hexdigest()
    print(json.dumps({"finalTargets": len(final_rows), "manifestSha256": checksum}), flush=True)

    print(json.dumps({"stagedValidation": stage_images({"validation": old["splits"]["validation"]}, scratch)}), flush=True)
    strong = strong_edit_count(old["splits"]["validation"], scratch)
    result = {
        "finalTargets": len(final_rows),
        "finalSeed": FINAL_SEED,
        "manifestSha256": checksum,
        "validationTargets": len(old["splits"]["validation"]),
        "strongValidationTargets": strong,
        "strongEditThreshold": STRONG_EDIT_THRESHOLD,
        "ready": strong >= 20,
    }
    (output / "preflight.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result), flush=True)
    if not result["ready"]:
        raise RuntimeError("Fewer than 20 strong-edit validation targets; revise protocol before GPU training")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/prepare_selector.py
"""Reserve unseen TST100K scenes before fitting a deployment selector.

This preflight reads only JSON manifests. It never fetches photographs, opens
expert edits, or loads either candidate matcher.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

from .data import DATASET_REVISION, fetch_manifest, scene_id
from .prepare_v4 import mounted_file

SEED = 7201
COUNTS = {"train": 1024, "validation": 256, "final": 256}


def scene_set(rows: list[dict]) -> set[str]:
    """Count a photograph in either target or reference position."""
    return {scene_id(row[key]) for row in rows for key in ("content", "reference")}


def select_rows(records: list[dict], excluded: set[str], counts: dict[str, int] = COUNTS,
                seed: int = SEED) -> dict:
    """Hash-rank eligible rows, keeping all three partitions scene-disjoint."""
    ranked = sorted(
        records,
        key=lambda row: hashlib.sha256(
            f"{seed}:{row['content']}:{row['reference']}:{row['gt']}".encode()
        ).digest(),
    )
    partitions = {name: [] for name in counts}
    occupied = set(excluded)
    for name, count in counts.items():
        targets = set()
        for row in ranked:
            target = scene_id(row["content"])
            reference = scene_id(row["reference"])
            if (target in occupied or reference in occupied or target == reference
                    or scene_id(row["gt"]) != target or target in targets):
                continue
            partitions[name].append(row)
            targets.add(target)
            if len(partitions[name]) == count:
                break
        if len(partitions[name]) != count:
            raise ValueError(f"Insufficient unseen {name} targets: {len(partitions[name])}/{count}")
        occupied.update(scene_set(partitions[name]))
    return partitions


def checksum(payload: dict) -> str:
    """Hash canonical JSON, independent of whitespace in the saved file."""
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode()).hexdigest()


def main() -> None:
    output = Path("/kaggle/working/colorcheck-tst100k-selector-preflight")
    output.mkdir(parents=True, exist_ok=True)
    old = json.loads(mounted_file("colorcheck-tst100k-expanded-validation", "splits.json").read_text())
    audit = json.loads(mounted_file("colorcheck-tst100k-fresh-evaluation", "splits.json").read_text())
    v4 = json.loads(mounted_file("colorcheck-tst100k-v4-preflight", "final-holdout.json").read_text())
    if old["datasetRevision"] != DATASET_REVISION or v4["datasetRevision"] != DATASET_REVISION:
        raise RuntimeError("Pinned dataset revision changed")
    excluded = scene_set([
        row for rows in old["splits"].values() for row in rows
    ] + audit["rows"] + v4["rows"])
    partitions = select_rows(fetch_manifest(), excluded)
    payload = {"datasetRevision": DATASET_REVISION, "seed": SEED, "splits": partitions}
    (output / "selector-splits.json").write_text(json.dumps(payload, indent=2))
    result = {
        "ready": True,
        "counts": {name: len(rows) for name, rows in partitions.items()},
        "excludedScenes": len(excluded),
        "manifestSha256": checksum(payload),
        "finalSha256": checksum({"rows": partitions["final"]}),
        "note": "Manifest-only preflight; no photographs or expert edits opened.",
    }
    (output / "preflight.json").write_text(json.dumps(result, indent=2))
    print(json.dumps(result), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/adaptive_rules_core.py
"""NumPy-only mirror of ColorCheck's browser matcher and pilot label search."""

from __future__ import annotations

import numpy as np

BASELINE = np.array([0.65, 0.45], dtype=np.float64)
STRENGTHS = (0.0, 0.15, 0.3, 0.45, 0.6, 0.65, 0.75, 0.9, 1.0)
FEATURE_NAMES = tuple(
    [f"source_mean_{channel}" for channel in "rgb"]
    + [f"reference_mean_{channel}" for channel in "rgb"]
    + [f"source_std_{channel}" for channel in "rgb"]
    + [f"reference_std_{channel}" for channel in "rgb"]
    + [f"mean_shift_{channel}" for channel in "rgb"]
    + [f"std_shift_{channel}" for channel in "rgb"]
)


def features(source: np.ndarray, reference: np.ndarray) -> np.ndarray:
    """Match web/src/adaptive-reference-match.js exactly in feature order."""
    source_mean = source.mean(axis=(1, 2))
    reference_mean = reference.mean(axis=(1, 2))
    source_std = source.std(axis=(1, 2))
    reference_std = reference.std(axis=(1, 2))
    return np.concatenate((
        source_mean, reference_mean, source_std, reference_std,
        reference_mean - source_mean, reference_std - source_std,
    )).astype(np.float64)


def match(source: np.ndarray, reference: np.ndarray, strengths: np.ndarray) -> np.ndarray:
    """Reproduce the site's bounded mean/spread transfer before manual edits."""
    source_mean = source.mean(axis=(1, 2))[:, None, None]
    reference_mean = reference.mean(axis=(1, 2))[:, None, None]
    source_std = source.std(axis=(1, 2))
    reference_std = reference.std(axis=(1, 2))
    gain = np.clip(reference_std / np.maximum(source_std, 0.015), 0.65, 1.5)[:, None, None]
    proposed = (source + (reference_mean - source_mean) * strengths[0]
                + (source - source_mean) * (gain - 1) * strengths[1])
    return np.clip(proposed, 0, 1)


def clipping(image: np.ndarray) -> float:
    return float(np.mean(np.any((image <= 1 / 255) | (image >= 254 / 255), axis=0)))


def training_label(source: np.ndarray, reference: np.ndarray, expert: np.ndarray) -> np.ndarray:
    """Find the best existing-control settings, with a mild clipping penalty."""
    best_score = float("inf")
    best = BASELINE
    original_clipping = clipping(source)
    for palette in STRENGTHS:
        for tone in STRENGTHS:
            strengths = np.array([palette, tone])
            result = match(source, reference, strengths)
            score = (float(np.mean(np.abs(result - expert)))
                     + 0.03 * max(0, clipping(result) - original_clipping)
                     + 0.003 * float(np.sum(np.abs(strengths - BASELINE))))
            if score < best_score:
                best_score, best = score, strengths
    return best


def gain_interval(values: np.ndarray, seed: int = 917) -> list[float]:
    generator = np.random.default_rng(seed)
    samples = values[generator.integers(0, len(values), size=(2000, len(values)))].mean(axis=1)
    return [float(value) for value in np.percentile(samples, (2.5, 97.5))]


In [ ]:
%%writefile /kaggle/working/tools/ml/tst100k/train_adaptive_rules.py
"""Fit browser-portable palette/tone rules from TST100K expert edits.

Only train and validation rows are opened. The final partition in the mounted
preflight is deliberately ignored. This is a pilot, not a release decision.
"""

from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from .adaptive_rules_core import (
    BASELINE, FEATURE_NAMES, STRENGTHS, clipping, features, gain_interval, match,
)
from .data import DATASET_REVISION, TripletDataset, stage_images
from .prepare_selector import checksum
from .prepare_v4 import mounted_file

PREFLIGHT = "colorcheck-tst100k-final-decision-preflight"
EXPECTED_MANIFEST_SHA256 = "b7c4b0a0977c2cdf6d998351d9fcc4a95d5b18eb4070defa6b6ac5637c1fee08"
COUNTS = {"train": 192, "validation": 64}
REGULARIZATION = (1.0, 10.0, 100.0)


def gpu_training_label(source: np.ndarray, reference: np.ndarray,
                       expert: np.ndarray) -> np.ndarray:
    """Score the fixed strength grid in parallel; the expert is never a model input."""
    device = torch.device("cuda")
    source_t, reference_t, expert_t = (
        torch.as_tensor(image, dtype=torch.float32, device=device)
        for image in (source, reference, expert)
    )
    values = torch.tensor(STRENGTHS, device=device)
    palette, tone = torch.meshgrid(values, values, indexing="ij")
    strengths = torch.stack((palette.flatten(), tone.flatten()), dim=1)
    source_mean = source_t.mean((1, 2))[:, None, None]
    reference_mean = reference_t.mean((1, 2))[:, None, None]
    gain = (reference_t.std((1, 2), unbiased=False)
            / source_t.std((1, 2), unbiased=False).clamp_min(0.015)).clamp(0.65, 1.5)
    proposed = (source_t[None]
                + (reference_mean - source_mean)[None] * strengths[:, 0, None, None, None]
                + ((source_t - source_mean) * (gain - 1)[:, None, None])[None]
                * strengths[:, 1, None, None, None]).clamp(0, 1)
    error = (proposed - expert_t).abs().mean((1, 2, 3))
    clipped = ((proposed <= 1 / 255) | (proposed >= 254 / 255)).any(dim=1).float().mean((1, 2))
    original = ((source_t <= 1 / 255) | (source_t >= 254 / 255)).any(dim=0).float().mean()
    baseline = torch.as_tensor(BASELINE, device=device)
    score = error + 0.03 * (clipped - original).clamp_min(0) \
        + 0.003 * (strengths - baseline).abs().sum(dim=1)
    return strengths[score.argmin()].cpu().numpy().astype(np.float64)


def rows_as_arrays(rows: list[dict], root: Path) -> list[tuple[np.ndarray, ...]]:
    """Small 64-pixel aligned images; never stage the full dataset locally."""
    dataset = TripletDataset(rows, root, 64)
    return [tuple(dataset[index][name].numpy().astype(np.float64)
                  for name in ("source", "reference", "expected"))
            for index in range(len(dataset))]


def predicted_strengths(feature: np.ndarray, scaler: StandardScaler, model: Ridge) -> np.ndarray:
    return np.clip(model.predict(scaler.transform(feature)), 0, 1)


def score_validation(
    images: list[tuple[np.ndarray, ...]],
    scaler: StandardScaler,
    model: Ridge,
) -> dict:
    """Use reference-free expert metrics only for evaluation, not prediction."""
    errors = {name: [] for name in ("adaptive", "baseline", "identity", "wrongReference")}
    clipping_rates = {name: [] for name in ("adaptive", "baseline")}
    predictions = []
    for index, (source, reference, expert) in enumerate(images):
        policy = predicted_strengths(features(source, reference)[None], scaler, model)[0]
        wrong_reference = images[(index + 1) % len(images)][1]
        wrong_policy = predicted_strengths(features(source, wrong_reference)[None], scaler, model)[0]
        candidates = {
            "adaptive": match(source, reference, policy),
            "baseline": match(source, reference, BASELINE),
            "identity": source,
            "wrongReference": match(source, wrong_reference, wrong_policy),
        }
        for name, result in candidates.items():
            errors[name].append(float(np.mean(np.abs(result - expert))))
        for name in clipping_rates:
            clipping_rates[name].append(clipping(candidates[name]))
        predictions.append(policy.tolist())
    gain = np.array(errors["baseline"]) - np.array(errors["adaptive"])
    summary = {
        "meanRgbMae": {name: float(np.mean(values)) for name, values in errors.items()},
        "meanClipping": {name: float(np.mean(values)) for name, values in clipping_rates.items()},
        "meanGainVsFixedMatcher": float(gain.mean()),
        "gainVsFixedCi95": gain_interval(gain),
        "fractionWithin002OfFixed": float(np.mean(gain >= -0.02)),
        "worstLossVsFixed": float(max(0, -gain.min())),
        "correctReferenceAdvantage": float(np.mean(errors["wrongReference"]) - np.mean(errors["adaptive"])),
        "meanStrengths": np.mean(predictions, axis=0).tolist(),
        "sampleCount": len(images),
    }
    summary["pilotGatesPass"] = bool(
        summary["meanGainVsFixedMatcher"] >= 0.002
        and summary["gainVsFixedCi95"][0] > 0
        and summary["fractionWithin002OfFixed"] >= 0.9
        and summary["worstLossVsFixed"] <= 0.08
        and summary["correctReferenceAdvantage"] > 0
        and summary["meanClipping"]["adaptive"] <= summary["meanClipping"]["baseline"] + 0.01
    )
    return summary


def main() -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("This GPU-assisted pilot requires CUDA")
    print({"gpu": torch.cuda.get_device_name(0)}, flush=True)
    output = Path("/kaggle/working/colorcheck-adaptive-rules")
    output.mkdir(parents=True, exist_ok=True)
    manifest = json.loads(mounted_file(PREFLIGHT, "splits.json").read_text())
    if manifest["datasetRevision"] != DATASET_REVISION or checksum(manifest) != EXPECTED_MANIFEST_SHA256:
        raise RuntimeError("Pinned training/validation manifest changed")
    selected = {name: manifest["splits"][name][:count] for name, count in COUNTS.items()}
    scratch = Path("/kaggle/working/colorcheck-adaptive-images")
    print(json.dumps({"staged": stage_images(selected, scratch)}), flush=True)
    train = rows_as_arrays(selected["train"], scratch)
    validation = rows_as_arrays(selected["validation"], scratch)
    train_features = np.stack([features(source, reference) for source, reference, _ in train])
    labels = np.stack([gpu_training_label(*images) for images in train])
    scaler = StandardScaler().fit(train_features)
    attempts = []
    for alpha in REGULARIZATION:
        model = Ridge(alpha=alpha).fit(scaler.transform(train_features), labels)
        attempts.append((score_validation(validation, scaler, model), alpha, model))
    # Choose by validation mean error only; the final partition remains unopened.
    summary, alpha, model = min(attempts, key=lambda row: row[0]["meanRgbMae"]["adaptive"])
    summary.update({"regularization": alpha, "trainingScenes": len(train),
                    "validationScenes": len(validation), "datasetRevision": DATASET_REVISION,
                    "manifestSha256": EXPECTED_MANIFEST_SHA256,
                    "note": "Pilot validation only; no final scenes opened or site deployment authorized."})
    package = {
        "featureNames": FEATURE_NAMES,
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "coefficients": model.coef_.tolist(),
        "intercepts": model.intercept_.tolist(),
        "researchOnly": True,
    }
    (output / "summary.json").write_text(json.dumps(summary, indent=2))
    (output / "adaptive-rules.json").write_text(json.dumps(package, indent=2))
    print(json.dumps(summary), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
import numpy as np, torch
from tools.ml.tst100k.adaptive_rules_core import training_label
from tools.ml.tst100k.train_adaptive_rules import gpu_training_label
assert torch.cuda.is_available(), 'Enable a GPU accelerator'
rng = np.random.default_rng(718)
for _ in range(4):
    source, reference, expert = rng.random((3, 3, 16, 16))
    np.testing.assert_allclose(gpu_training_label(source, reference, expert), training_label(source, reference, expert))
print('GPU label search matches the NumPy reference')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'tools.ml.tst100k.train_adaptive_rules'], cwd='/kaggle/working', check=True, timeout=5400)
